# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration of the FAIR² dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

_This notebook explores tabular clinical data on second primary colorectal cancer in cancer survivors, including clinical, anatomical, and molecular variables._

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
We first load the metadata and access the dataset resources using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the dataset from the Croissant URL
dataset = mlc.Dataset(croissant_url)

# Fetch the metadata (as an object, not dict!)
md = dataset.metadata
print(f"{md.name}: {md.description}\nPublished: {md.datePublished} (v{md.version})\nIdentifier: {md.identifier}")

## 2. Data Overview
Let's review available record sets, their field `@id`s, and columns. All references to data will use the resource `@id` fields from the Croissant schema.

In [ ]:
# List information about all record sets, fields, and columns via their '@id's
print("Available Record Sets:")
record_set_objs = dataset.record_sets  # List of mlcroissant.RecordSet objects
for rs in record_set_objs:
    print(f"\n- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    if rs.fields:
        print(f"  Fields:")
        for f in rs.fields:
            print(f"    - Field @id: {f.id}")
            print(f"      Name: {f.name}")
            print(f"      Data type: {f.dataType}")
            if hasattr(f, 'source') and hasattr(f.source, 'id'):
                print(f"      Source (column @id): {f.source.id}")

## 3. Data Extraction
Let's extract data from a record set of interest. Update `<record_set_id>` with one of the `@id`s printed above. All columns will be included as present in the schema.

In [ ]:
# For this dataset, there's typically one record set (main data table).
# We'll extract the first available RecordSet by its @id. Use the overview cell to get additional available ids.

record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))  # Each record is a dict
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records.")
    else:
        print("No records loaded for this record set.")

# Display column names for the primary record set.
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in RecordSet {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll now process the main data table:
- Filter on a numeric field (e.g., age at 2nd primary CRC, if present)
- Normalize this numeric field
- Group by an anatomical/categorical field (e.g., the anatomical site of CRC)

Please use the field `@id`s and names from your dataset printed above. Below are example variable assignments based on typical clinical datasets.

In [ ]:
# Choose your actual field and group @ids from the previous overview output!
# EXAMPLES: Replace with the @ids that correspond to the numeric and group fields below:
# For demonstration, we try to infer possible field names, else show the table head only.

df = dataframes[main_rs_id]

# Try to guess an age/numeric field from possible variants
possible_numeric_fields = [c for c in df.columns if 'age' in c.lower() or 'interval' in c.lower() or 'number' in c.lower()]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns)>0 else df.columns[0]

# Try to use anatomical or sex/grouping field
possible_group_fields = [c for c in df.columns if 'anatomic' in c.lower() or 'site' in c.lower() or 'location' in c.lower() or 'sex' in c.lower() or 'gender' in c.lower()]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
else:
    group_field_id = df.columns[0]

print(f"Numeric field selected: {numeric_field_id}")
print(f"Group (categorical) field selected: {group_field_id}")

# Filtering: Only keep rows with numeric field > a threshold (e.g. 50)
try:
    threshold = 50
    filtered_df = df[ pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold ]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
except Exception as e:
    print("Data inspection/processing failed:", str(e))
    print("\nShowing data head:")
    print(df.head())

## 5. Visualization
Now we plot the distribution of the selected numeric field and boxplot/group means by the selected categorical field (`@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(df[numeric_field_id].dropna()) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=pd.to_numeric(df[numeric_field_id], errors='coerce'), data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and process a real-world clinical dataset (FAIR², Croissant format) using `mlcroissant`. We used the Croissant schema `@id` references to all entities, reviewed main fields, performed EDA (filtering, normalization, group analysis), and visualized key patterns. For more complex analyses or publication-quality statistics, refer to the dataset's detailed dictionary and schema at the provided Croissant URL.